# RE1.1 · Auditoría del protocolo de curación

Aplica el protocolo de `src/data/annotations.py` regla por regla sobre las tablas crudas de
`data/raw/` y cuenta qué toca cada una: es la evidencia del documento `RE_1-1_protocolo-curacion.tex`.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd
import reporte
from reporte import PROJECT_DIR, number, percent
from slugify import slugify

from core.config import RAW_DIR
from data.annotations import (
    CALL_SYNONYMS,
    DROP_COLUMNS,
    MANUAL_FIXES,
    MANUAL_SYNONYMS,
    MAX_FREQ_HZ,
    MIN_DURATION_S,
    NOISE,
    audio_signature,
    cleaned,
    list_files,
    recording_stem,
    species_of,
)
from data.species import BACKGROUND_SPECIES, CALL_TYPES, VALID_PAIRS, Species

OUT = reporte.out_dir("RE_1-1")
RAW_SPECIES, RAW_CALL = "species", "call_type"

In [2]:
def recording_of(table: Path) -> Path | None:
    for suffix in (".wav", ".WAV"):
        candidate = table.with_name(recording_stem(table) + suffix)
        if candidate.is_file():
            return candidate
    return None


tables = [
    path for path in list_files(RAW_DIR, ".txt") if species_of(path) not in BACKGROUND_SPECIES
]
recordings = [
    path
    for path in list_files(RAW_DIR, ".wav") + list_files(RAW_DIR, ".WAV")
    if species_of(path) not in BACKGROUND_SPECIES
]
paired = {table: audio for table in tables if (audio := recording_of(table)) is not None}
orphans = [table for table in tables if table not in paired]
inventory = (
    pd.DataFrame(
        {
            "grabaciones": Counter(species_of(p).upper() for p in recordings),
            "tablas": Counter(species_of(p).upper() for p in tables),
            "tablas con audio": Counter(species_of(p).upper() for p in paired),
        }
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)
inventory.loc["Total"] = inventory.sum()
inventory

,grabaciones,tablas,tablas con audio
AA,117,85,84
AC,561,388,385
AS,808,780,779
CC,16,8,7
LW,530,411,410
PT,375,321,320
SB,290,257,256
SM,659,585,582
Total,3356,2835,2823


In [3]:
# Cada tabla cruda pasa por las reglas en el orden de `clean_annotations`; se cuenta qué toca cada una.
def audit(table: Path, species: str) -> tuple[pd.DataFrame, pd.DataFrame, Counter]:
    raw = pd.read_csv(table, sep="\t")
    hits: Counter = Counter(filas=len(raw))
    hits.update({f"col:{c}": 1 for c in raw.columns})
    df = raw.copy()
    df.columns = [cleaned(c) for c in df.columns]
    manual = df[[RAW_SPECIES, RAW_CALL]].copy()
    df = df.drop(columns=DROP_COLUMNS, errors="ignore")

    typed = df[RAW_CALL].map(lambda v: v if isinstance(v, str) else None)
    slug = typed.map(lambda v: slugify(v, separator="_") or None if isinstance(v, str) else None)
    # Más allá de la mayúscula: espacios, signos y separadores que la slugificación absorbe
    hits["tipografía"] = int((typed.str.strip().str.lower().fillna("") != slug.fillna("")).sum())
    synonyms = MANUAL_SYNONYMS | CALL_SYNONYMS.get(species, {})
    mapped = slug.replace(synonyms)
    hits["sinónimo"] = int((slug.fillna("") != mapped.fillna("")).sum())
    hits["vacío"] = int(mapped.isna().sum())
    hits["ruido"] = int(mapped.eq(NOISE).sum())
    df[RAW_CALL] = mapped

    written = df[RAW_SPECIES].map(
        lambda v: slugify(v, separator="_") if isinstance(v, str) else None
    )
    hits["especie vacía"] = int(written.isna().sum())
    hits["especie≠carpeta"] = int((written.notna() & written.ne(species)).sum())
    df[RAW_SPECIES] = species
    df = df.loc[df[RAW_CALL].notna() & df[RAW_CALL].ne(NOISE)]

    for (bad_sp, bad_ct), (sp, ct) in MANUAL_FIXES.items():
        wrong = df[RAW_SPECIES].eq(bad_sp) & df[RAW_CALL].eq(bad_ct)
        hits["par imposible"] += int(wrong.sum())
        df.loc[wrong, [RAW_SPECIES, RAW_CALL]] = [sp, ct]

    pairs = pd.Series(list(zip(df[RAW_SPECIES], df[RAW_CALL], strict=True)), index=df.index)
    df["requires_review"] = ~pairs.isin(VALID_PAIRS)
    hits["fuera de vocabulario"] = int(df["requires_review"].sum())

    hits["sobre Nyquist"] = int((df["high_freq_hz"] > MAX_FREQ_HZ).sum())
    df["high_freq_hz"] = df["high_freq_hz"].clip(upper=MAX_FREQ_HZ)
    df["duration_s"] = df["end_time_s"] - df["begin_time_s"]
    df["bandwidth_hz"] = df["high_freq_hz"] - df["low_freq_hz"]
    degenerate = (df["duration_s"] < MIN_DURATION_S) | (df["bandwidth_hz"] <= 0)
    hits["degenerada"] = int(degenerate.sum())
    df = df.loc[~degenerate]
    hits["conservadas"] = len(df)
    return df.assign(audio_path=str(recording_of(table)), source=str(table)), manual, hits


frames, manuals, counts = [], [], Counter()
failed = []
for table, audio in paired.items():
    try:
        frame, manual, hits = audit(table, species_of(audio))
    except Exception as exc:  # una tabla ilegible no detiene la auditoría, igual que en el pipeline
        failed.append(f"{table.relative_to(RAW_DIR)}: {exc}")
        continue
    frames.append(frame)
    manuals.append(manual)
    counts.update(hits)
annotations = pd.concat(frames, ignore_index=True)
raw_manual = pd.concat(manuals, ignore_index=True)
print(f"{len(paired)} tablas · {len(failed)} ilegibles · {counts['filas']} filas crudas")
failed

2823 tablas · 4 ilegibles · 20082 filas crudas


['howler_monkey__AS/20240120_065845.txt: No columns to parse from file',
 "night_monkey__AA/20240609_174227.txt: 'utf-8' codec can't decode byte 0x80 in position 37: invalid start byte",
 "night_monkey__AA/20240609_174409.txt: 'utf-8' codec can't decode byte 0x80 in position 37: invalid start byte",
 'peruvian_spider_monkey__AC/20240409_064044.txt: No columns to parse from file']

In [4]:
# Copias byte-idénticas del mismo wav en varias carpetas de especie: una sola grabación con la
# unión de sus anotaciones (`unify_copies`).
signatures = {path: audio_signature(Path(path)) for path in annotations["audio_path"].unique()}
canonical = {}
for path, signature in sorted(signatures.items()):
    canonical[path] = canonical.get(next(p for p, s in signatures.items() if s == signature), path)
groups = pd.Series(canonical).groupby(lambda p: canonical[p]).size()
unified = annotations.assign(audio_path=annotations["audio_path"].map(canonical))
n_duplicated_rows = int(
    unified.duplicated(subset=[c for c in unified.columns if c != "source"]).sum()
)
copies = pd.Series(
    {
        "grabaciones con copia": int((groups > 1).sum()),
        "copias unidas": int((groups - 1).clip(lower=0).sum()),
        "grabaciones tras unir": len(groups),
        "anotaciones repetidas descartadas": n_duplicated_rows,
    }
)
copies

grabaciones con copia                 120
copias unidas                         128
grabaciones tras unir                2664
anotaciones repetidas descartadas       0
dtype: int64

In [5]:
headers = {k[4:]: v for k, v in counts.items() if k.startswith("col:")}
rows = [
    ("Tablas sin grabación homónima", len(orphans), "se omiten"),
    ("Tablas ilegibles", len(failed), "se omiten y se registran"),
    ("Filas crudas en tablas con audio", counts["filas"], "punto de partida"),
    ("Tipo de llamada con espacios o signos de más", counts["tipografía"], "se normaliza"),
    ("Tipo de llamada por sinónimo o nombre legible", counts["sinónimo"], "mapa explícito"),
    ("Especie vacía", counts["especie vacía"], "la de la carpeta"),
    ("Especie escrita distinta de la carpeta", counts["especie≠carpeta"], "prevalece la carpeta"),
    ("Tipo de llamada vacío", counts["vacío"], "se descarta"),
    ("Tipo de llamada \\texttt{noise}", counts["ruido"], "se descarta"),
    ("Par imposible corregido", counts["par imposible"], "\\texttt{aa/hc} $\\to$ \\texttt{aa/hm}"),
    (
        "Frecuencia superior por encima de Nyquist",
        counts["sobre Nyquist"],
        "se recorta a 22\\,050\\,Hz",
    ),
    ("Caja degenerada ($<10$\\,ms o sin banda)", counts["degenerada"], "se descarta"),
    ("Par fuera de vocabulario", counts["fuera de vocabulario"], "\\texttt{requires\\_review}"),
    (
        "Copias del mismo audio en otra carpeta",
        int(copies["copias unidas"]),
        "se unen a su original",
    ),
    ("Anotaciones repetidas entre copias", n_duplicated_rows, "se descartan"),
    ("Anotaciones conservadas", len(unified) - n_duplicated_rows, "$\\to$ \\texttt{data/cleaned/}"),
]
reporte.write_rows(OUT / "reglas.tex", [(rule, number(n), how) for rule, n, how in rows])
pd.DataFrame(rows, columns=["regla", "filas", "tratamiento"])

,regla,filas,tratamiento
0,Tablas sin grabación homónima,12,se omiten
1,Tablas ilegibles,4,se omiten y se registran
2,Filas crudas en tablas con audio,20082,punto de partida
3,Tipo de llamada con espacios o signos de más,52,se normaliza
4,Tipo de llamada por sinónimo o nombre legible,105,mapa explícito
5,Especie vacía,879,la de la carpeta
6,Especie escrita distinta de la carpeta,129,prevalece la carpeta
7,Tipo de llamada vacío,180,se descarta
8,Tipo de llamada \texttt{noise},866,se descarta
9,Par imposible corregido,23,\texttt{aa/hc} $\to$ \texttt{aa/hm}


In [6]:
# Cuántos valores distintos quedan en cada columna manual a medida que avanza la normalización.
def unique_values(series: pd.Series) -> int:
    return int(series.dropna().map(str).nunique())


def slugged(series: pd.Series) -> pd.Series:
    return series.map(lambda v: slugify(v, separator="_") if isinstance(v, str) else None)


stages = pd.DataFrame(
    {
        "etapa": ["crudo", "tipografía", "sinónimos", "vocabulario"],
        "especie": [
            unique_values(raw_manual[RAW_SPECIES]),
            unique_values(slugged(raw_manual[RAW_SPECIES])),
            len(inventory) - 1,
            len(inventory) - 1,
        ],
        "llamada": [
            unique_values(raw_manual[RAW_CALL]),
            unique_values(slugged(raw_manual[RAW_CALL])),
            unique_values(annotations[RAW_CALL]),
            unique_values(annotations.loc[~annotations["requires_review"], RAW_CALL]),
        ],
    }
)
reporte.write_rows(OUT / "etapas.tex", stages.to_numpy().tolist())
stages

,etapa,especie,llamada
0,crudo,19,63
1,tipografía,15,55
2,sinónimos,8,42
3,vocabulario,8,31


In [7]:
# El vocabulario cerrado y el mapa de sinónimos, tal como los declara el código.
def readable(code: str) -> str:
    return code.replace("_", " ")


reporte.write_rows(
    OUT / "vocabulario.tex",
    [
        (
            species.name,
            len(codes),
            ", ".join(f"\\texttt{{{code}}} ({readable(name)})" for code, name in codes.items()),
        )
        for species, codes in CALL_TYPES.items()
        if species.name.lower() not in BACKGROUND_SPECIES
    ],
)
reporte.write_rows(
    OUT / "sinonimos.tex",
    [
        (f"\\texttt{{{k.replace('_', '\\_')}}}", f"\\texttt{{{v}}}")
        for k, v in MANUAL_SYNONYMS.items()
        if k != "avevoc"
    ],
)
n_vocab = sum(len(c) for s, c in CALL_TYPES.items() if s is not Species.AV)
print(f"{n_vocab} pares en el vocabulario · {len(MANUAL_SYNONYMS) - 1} sinónimos manuales")

41 pares en el vocabulario · 10 sinónimos manuales


In [8]:
review = (
    annotations.loc[annotations["requires_review"]]
    .groupby([RAW_SPECIES, RAW_CALL])
    .size()
    .sort_values(ascending=False)
)
cells = [(f"\\texttt{{{sp}/{ct}}}", number(n)) for (sp, ct), n in review.items()]
half = (len(cells) + 1) // 2  # a dos columnas
cells += [("", "")] * (2 * half - len(cells))
reporte.write_rows(
    OUT / "fuera_vocabulario.tex", [a + b for a, b in zip(cells[:half], cells[half:])]
)
reporte.write_macros(
    OUT / "valores.tex",
    {
        "n_recordings": number(len(recordings)),
        "n_tables": number(len(tables)),
        "n_paired": number(len(paired)),
        "n_orphans": number(len(orphans)),
        "n_failed": number(len(failed)),
        "n_raw_rows": number(counts["filas"]),
        "n_kept": number(len(unified) - n_duplicated_rows),
        "pct_kept": percent((len(unified) - n_duplicated_rows) / counts["filas"]),
        "n_review": number(counts["fuera de vocabulario"]),
        "n_review_pairs": number(len(review)),
        "n_noise": number(counts["ruido"]),
        "n_nyquist": number(counts["sobre Nyquist"]),
        "n_degenerate": number(counts["degenerada"]),
        "n_species_mismatch": number(counts["especie≠carpeta"]),
        "n_species_empty": number(counts["especie vacía"]),
        "n_typography": number(counts["tipografía"]),
        "n_synonyms": number(counts["sinónimo"]),
        "n_fixed": number(counts["par imposible"]),
        "n_empty_call": number(counts["vacío"]),
        "n_copies": number(int(copies["copias unidas"])),
        "n_unique_recordings": number(len(groups)),
        "n_raw_call_values": number(stages.llamada[0]),
        "n_vocab_call_values": number(stages.llamada[3]),
        "n_headers": number(len(headers)),
        "n_vocab": number(n_vocab),
    },
)
review

species  call_type
as       cp           91
         pp           85
         ip           73
lw       a            20
         b            15
sm       hc            8
sb       pcs           8
aa       sqc           6
sm       sic           4
as       cc            4
         pc            3
sb       phc           3
lw       c             3
sm       chc           3
aa       hf            1
as       chc           1
pt       d             1
         chc           1
lw       t             1
         lw            1
         chc           1
sb       hic           1
         cc            1
sm       spc           1
dtype: int64